In [18]:
import pandas as pd
import matplotlib.pyplot as plt

data = pd.read_csv('cdc_brfss.csv')
print(data.shape)
data.head()

(253680, 22)


,Diabetes_binary,HighBP,HighChol,CholCheck,BMI,Smoker,Stroke,HeartDiseaseorAttack,PhysActivity,Fruits,...,AnyHealthcare,NoDocbcCost,GenHlth,MentHlth,PhysHlth,DiffWalk,Sex,Age,Education,Income
0,0.0,1.0,1.0,1.0,40.0,1.0,0.0,0.0,0.0,0.0,...,1.0,0.0,5.0,18.0,15.0,1.0,0.0,9.0,4.0,3.0
1,0.0,0.0,0.0,0.0,25.0,1.0,0.0,0.0,1.0,0.0,...,0.0,1.0,3.0,0.0,0.0,0.0,0.0,7.0,6.0,1.0
2,0.0,1.0,1.0,1.0,28.0,0.0,0.0,0.0,0.0,1.0,...,1.0,1.0,5.0,30.0,30.0,1.0,0.0,9.0,4.0,8.0
3,0.0,1.0,0.0,1.0,27.0,0.0,0.0,0.0,1.0,1.0,...,1.0,0.0,2.0,0.0,0.0,0.0,0.0,11.0,3.0,6.0
4,0.0,1.0,1.0,1.0,24.0,0.0,0.0,0.0,1.0,1.0,...,1.0,0.0,2.0,3.0,0.0,0.0,0.0,11.0,5.0,4.0


In [19]:
# Drop NaN value
print("Any null value:", any(data.isnull()))
print("Any NaN value:", any(data.isna()))
print("Before Droping NaN Number of Rows:", len(data))

data = data.dropna()
data = data.drop_duplicates()
print("After Droping NaN Number of Rows and Duplicates:", len(data))

# Move 'Diabetes_binary' column to the end of dataframe
data['Diabetes_binary'] = data.pop('Diabetes_binary')

data.tail()

Any null value: True
Any NaN value: True
Before Droping NaN Number of Rows: 253680
After Droping NaN Number of Rows and Duplicates: 229474


,HighBP,HighChol,CholCheck,BMI,Smoker,Stroke,HeartDiseaseorAttack,PhysActivity,Fruits,Veggies,...,NoDocbcCost,GenHlth,MentHlth,PhysHlth,DiffWalk,Sex,Age,Education,Income,Diabetes_binary
253675,1.0,1.0,1.0,45.0,0.0,0.0,0.0,0.0,1.0,1.0,...,0.0,3.0,0.0,5.0,0.0,1.0,5.0,6.0,7.0,0.0
253676,1.0,1.0,1.0,18.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,4.0,0.0,0.0,1.0,0.0,11.0,2.0,4.0,1.0
253677,0.0,0.0,1.0,28.0,0.0,0.0,0.0,1.0,1.0,0.0,...,0.0,1.0,0.0,0.0,0.0,0.0,2.0,5.0,2.0,0.0
253678,1.0,0.0,1.0,23.0,0.0,0.0,0.0,0.0,1.0,1.0,...,0.0,3.0,0.0,0.0,0.0,1.0,7.0,5.0,1.0,0.0
253679,1.0,1.0,1.0,25.0,0.0,0.0,1.0,1.0,1.0,0.0,...,0.0,2.0,0.0,0.0,0.0,0.0,9.0,6.0,2.0,1.0


In [20]:
cdc_corr = data.corr()
print(cdc_corr.iloc[-1,:].sort_values(ascending=False))

Diabetes_binary         1.000000
GenHlth                 0.276940
HighBP                  0.254318
DiffWalk                0.205302
BMI                     0.205086
HighChol                0.194944
Age                     0.177263
HeartDiseaseorAttack    0.168213
PhysHlth                0.156211
Stroke                  0.099193
CholCheck               0.072523
MentHlth                0.054153
Smoker                  0.045504
Sex                     0.032724
AnyHealthcare           0.025331
NoDocbcCost             0.020048
Fruits                 -0.024805
Veggies                -0.041734
HvyAlcoholConsump      -0.065950
PhysActivity           -0.100404
Education              -0.102686
Income                 -0.140659
Name: Diabetes_binary, dtype: float64


In [21]:
#split

from sklearn.model_selection import train_test_split

def data_split(data):
    
    X = data.iloc[:,:-1]
    y = data.iloc[:,-1]
    
    train_x, temp_x, train_y, temp_y = train_test_split(X,y,test_size = 0.3, random_state=0)

    test_x, val_x, test_y, val_y = train_test_split(temp_x, temp_y, test_size=0.3333, random_state=0, stratify=temp_y)


# Check sizes
    print("Train size:", len(train_x))
    print("Test size:", len(test_x))
    print("Validation size:", len(val_x))
    
    return train_x, test_x, train_y, test_y, val_x, val_y

train_x, test_x, train_y, test_y, val_x, val_y = data_split(data)

Train size: 160631
Test size: 45897
Validation size: 22946


In [22]:

train_df = pd.concat([train_x, train_y], axis=1)
test_df  = pd.concat([test_x, test_y], axis=1)
val_df   = pd.concat([val_x, val_y], axis=1)

train_df.to_csv("cdc_train.csv", index=False)
test_df.to_csv("cdc_test.csv", index=False)
val_df.to_csv("cdc_validation.csv", index=False)


In [23]:
#extratrees
from sklearn.ensemble import ExtraTreesClassifier
from sklearn.model_selection import cross_val_score

model = ExtraTreesClassifier(random_state = 0)

model.fit(train_x, train_y)

cross_val_score(model, train_x, train_y, scoring = 'accuracy', cv = 5, n_jobs = 1).mean()

0.8363329696073045

In [24]:
#catmod

from catboost import CatBoostClassifier
from sklearn.metrics import accuracy_score

cat_mod = CatBoostClassifier(
    iterations=100,      
    learning_rate=0.1,   
    depth=6,              
    verbose=0             
)

cat_mod.fit(train_x, train_y)
y_pred = cat_mod.predict(test_x)
accuracy = accuracy_score(test_y, y_pred)
print("Accuracy:", accuracy_score(test_y, y_pred))

Accuracy: 0.8545438699697148


In [25]:
#adaboost
from sklearn.ensemble import AdaBoostClassifier

ada = AdaBoostClassifier(n_estimators=50,
                         learning_rate=1)

ada_model = ada.fit(train_x, train_y)

y_pred = ada_model.predict(test_x)

print("Accuracy:", accuracy_score(test_y, y_pred))

Accuracy: 0.8506438329302569


In [26]:
from c50py import C5Classifier

c5 = C5Classifier(trials=1)

c5_model = c5.fit(train_x, train_y)

y_pred = c5_model.predict(test_x)

print("Accuracy:", accuracy_score(test_y, y_pred))

Accuracy: 0.8468309475564851


In [17]:
# import c50py
# print(c50py)
# print(dir(c50py))